# 09 — Yoko LAS → DSM registration (Musanga Final)

Uses the **exact supplied locations** under `/content/drive/MyDrive/Colab Notebooks/Musanga/Final`:

- Input: `Data/LAS/raw/Yoko.las`
- Reference DSM: `Data/DSM/Yoko_kvo_georef_DSM.tif`
- Output: `Data/Las/Yoko_aligned.las`

No searching or automatic folder discovery. Inspect the QA plots before accepting the alignment.

**RAM-safe revision:** The reference DSM is downsampled by GDAL on read; the notebook never loads its full-resolution window into Python memory. The LAS is processed in chunks.


In [ ]:
!pip -q install laspy lazrs rasterio scipy pyproj pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import copy, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio, laspy
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from scipy.ndimage import map_coordinates
from pyproj import CRS

ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
LAS_REL = 'Data/LAS/raw/Yoko.las'
DSM_REL = 'Data/DSM/Yoko_kvo_georef_DSM.tif'
OUTPUT_REL = 'Data/Las'
SEARCH_RADIUS_M = 12.0
COARSE_STEP_M = 2.0
FINE_STEP_M = 0.5
GRID_RES_M = 2.0
MAX_GRID_CELLS = 2_000_000
MAX_MATCH_CELLS = 30_000
CHUNK_POINTS = 200_000
MIN_MATCH_CELLS = 300
APPLY_Z = False
EXPORT_CORRECTED_LAS = True
SEED = 9

In [ ]:
# Follow the same fixed ROOT / data / Geo_Ref convention as the previous Musanga notebooks.
# No recursive searching of Drive, no file discovery.
root = ROOT
las_path = ROOT / LAS_REL
dsm_path = ROOT / DSM_REL
outdir = ROOT / OUTPUT_REL

if not root.is_dir():
    raise FileNotFoundError(f'Musanga project root missing: {root}')
if not las_path.is_file():
    raise FileNotFoundError(f'LAS missing: {las_path}')
if not dsm_path.is_file():
    raise FileNotFoundError(
        f'DSM missing: {dsm_path}. The reference DSM path is fixed to Final/Data/DSM; '
        'check DSM_REL if this DSM is stored elsewhere.'
    )
outdir.mkdir(parents=True, exist_ok=True)
print('LAS:', las_path)
print('DSM:', dsm_path)
print('Output:', outdir / 'Yoko_aligned.las')

In [ ]:
with laspy.open(las_path) as reader:
    h=reader.header
    npoints=h.point_count
    xyz_min,xyz_max=h.mins.copy(),h.maxs.copy()
    las_crs=h.parse_crs()
with rasterio.open(dsm_path) as ds:
    dsm_crs, dbounds, dres=ds.crs,ds.bounds,ds.res
    dsm_shape=ds.shape
print('LAS:',f'{npoints:,}','points; CRS',las_crs,'bounds',xyz_min,xyz_max)
print('DSM:',dsm_crs,'pixel',dres,'shape',dsm_shape,'bounds',dbounds)
if las_crs is None or dsm_crs is None:
    raise ValueError('Resolve missing CRS before attempting registration.')
if not CRS.from_user_input(las_crs).equals(CRS.from_user_input(dsm_crs)):
    raise ValueError('CRSs differ: reproject LAS first; translation is not reprojection.')
if not dsm_crs.is_projected:
    raise ValueError('Requires projected metric CRS.')
units=[a.unit_conversion_factor for a in CRS.from_user_input(dsm_crs).axis_info[:2]]
if any(abs(float(u)-1)>1e-6 for u in units):
    raise ValueError('Requires metre coordinates.')
ox=max(0,min(xyz_max[0],dbounds.right)-max(xyz_min[0],dbounds.left))
oy=max(0,min(xyz_max[1],dbounds.top)-max(xyz_min[1],dbounds.bottom))
print('Bounding-box overlap (m):',round(ox,1),round(oy,1))
if ox==0 or oy==0:
    raise ValueError('No overlap; check coordinates/CRS before registration.')

In [ ]:
xmin,ymin,xmax,ymax=xyz_min[0],xyz_min[1],xyz_max[0],xyz_max[1]
resolution=float(max(GRID_RES_M,*map(abs,dres)))
while np.ceil((xmax-xmin)/resolution)*np.ceil((ymax-ymin)/resolution)>MAX_GRID_CELLS:
    resolution*=1.5
cols=int(np.ceil((xmax-xmin)/resolution))
rows=int(np.ceil((ymax-ymin)/resolution))
print('LAS grid:',rows,cols,'cell size',resolution)
max_flat=np.full(rows*cols,-np.inf,dtype=np.float32)
with laspy.open(las_path) as reader:
    for batch in reader.chunk_iterator(CHUNK_POINTS):
        x=np.asarray(batch.x);y=np.asarray(batch.y);z=np.asarray(batch.z)
        c=np.floor((x-xmin)/resolution).astype(np.int64)
        r=np.floor((ymax-y)/resolution).astype(np.int64)
        ok=np.isfinite(z)&(c>=0)&(c<cols)&(r>=0)&(r<rows)
        np.maximum.at(max_flat,r[ok]*cols+c[ok],z[ok].astype(np.float32))
las_grid=max_flat.reshape(rows,cols)
las_grid[~np.isfinite(las_grid)]=np.nan
# Read DSM through GDAL reprojection into a bounded coarse array.
# NEVER load a full-resolution DSM window into RAM.
margin = SEARCH_RADIUS_M + 3 * resolution
target_left = xmin - margin
target_top = ymax + margin
target_right = xmax + margin
target_bottom = ymin - margin
target_cols = int(np.ceil((target_right - target_left) / resolution))
target_rows = int(np.ceil((target_top - target_bottom) / resolution))
while target_cols * target_rows > MAX_GRID_CELLS:
    resolution *= 1.5
    # The LAS grid and DSM target must share this final resolution.
    # Rather than silently mismatch grids, stop with a clear diagnostic.
    raise MemoryError(
        'DSM target footprint exceeds MAX_GRID_CELLS after LAS-grid setup. '
        'Increase GRID_RES_M or reduce spatial extent before rerunning.'
    )
transform = from_origin(target_left, target_top, resolution, resolution)
dsm = np.full((target_rows, target_cols), np.nan, dtype=np.float32)
with rasterio.open(dsm_path) as ds:
    reproject(
        source=rasterio.band(ds, 1),
        destination=dsm,
        src_transform=ds.transform,
        src_crs=ds.crs,
        src_nodata=ds.nodata,
        dst_transform=transform,
        dst_crs=ds.crs,
        dst_nodata=np.nan,
        resampling=Resampling.average,
        num_threads=2,
        warp_mem_limit=64,
        init_dest_nodata=True,
    )
print(f'DSM resampled directly to {target_rows}x{target_cols} '
      f'({dsm.nbytes / 2**20:.1f} MiB in RAM) at {resolution:.2f} m')
print('Occupied LAS grid cells:',np.isfinite(las_grid).sum(),'DSM window:',dsm.shape)

In [ ]:
rr,cc=np.where(np.isfinite(las_grid))
if len(rr)<MIN_MATCH_CELLS:
    raise RuntimeError('Too few valid LAS cells to register')
rng=np.random.default_rng(SEED)
if len(rr)>MAX_MATCH_CELLS:
    take=rng.choice(len(rr),MAX_MATCH_CELLS,replace=False)
    rr,cc=rr[take],cc[take]
surface_z=las_grid[rr,cc].astype(np.float64)
px=xmin+(cc+0.5)*resolution
py=ymax-(rr+0.5)*resolution

def residuals(dx,dy):
    dcol=(px+dx-transform.c)/transform.a-0.5
    drow=(py+dy-transform.f)/transform.e-0.5
    z=map_coordinates(dsm,[drow,dcol],order=1,mode='constant',cval=np.nan,prefilter=False)
    good=np.isfinite(z)&np.isfinite(surface_z)
    return z[good]-surface_z[good],good

def score(dx,dy):
    r,good=residuals(dx,dy)
    if len(r)<MIN_MATCH_CELLS or good.mean()<0.60:
        return np.inf
    return float(np.median(np.abs(r-np.median(r))))

steps=np.arange(-SEARCH_RADIUS_M,SEARCH_RADIUS_M+COARSE_STEP_M*0.25,COARSE_STEP_M)
coarse=pd.DataFrame([(float(dx),float(dy),score(dx,dy)) for dy in steps for dx in steps],
                    columns=['dx','dy','mad_m'])
if not np.isfinite(coarse.mad_m.min()):
    raise RuntimeError('No matching surface overlap. Check data or search radius.')
b=coarse.loc[coarse.mad_m.idxmin()]
fx=np.arange(max(-SEARCH_RADIUS_M,b.dx-COARSE_STEP_M),min(SEARCH_RADIUS_M,b.dx+COARSE_STEP_M)+1e-6,FINE_STEP_M)
fy=np.arange(max(-SEARCH_RADIUS_M,b.dy-COARSE_STEP_M),min(SEARCH_RADIUS_M,b.dy+COARSE_STEP_M)+1e-6,FINE_STEP_M)
fine=pd.DataFrame([(float(dx),float(dy),score(dx,dy)) for dy in fy for dx in fx],
                  columns=['dx','dy','mad_m'])
b=fine.loc[fine.mad_m.idxmin()]
dx,dy=float(b.dx),float(b.dy)
before,g0=residuals(0,0)
after,g1=residuals(dx,dy)
dz=float(np.median(after))
print(f'Estimated shift LAS: dX={dx:+.2f} m, dY={dy:+.2f} m; optional dZ={dz:+.2f} m')
print(f'Centered MAD: {score(0,0):.3f} -> {score(dx,dy):.3f} m')
print('Matched cells:',len(before),'->',len(after))
if abs(dx)>=SEARCH_RADIUS_M-0.01 or abs(dy)>=SEARCH_RADIUS_M-0.01:
    warnings.warn('Optimum on search boundary: expand range and rerun.')
if score(dx,dy)>=0.98*score(0,0):
    warnings.warn('Less than 2% improvement: alignment uncertain.')

In [ ]:
pivot=coarse.pivot(index='dy',columns='dx',values='mad_m')
fig,axs=plt.subplots(1,3,figsize=(17,4.5))
im=axs[0].imshow(pivot.values,origin='lower',
                 extent=[steps.min(),steps.max(),steps.min(),steps.max()],aspect='equal')
axs[0].scatter([dx],[dy],marker='x',s=120,c='red')
axs[0].set(xlabel='dX (m)',ylabel='dY (m)',title='Centered MAD (m)')
fig.colorbar(im,ax=axs[0],label='m')
for ax,res,title in [(axs[1],before,'Before'),(axs[2],after,'After XY')]:
    ax.hist(res,bins=100,range=np.percentile(res,[2,98]))
    ax.axvline(np.median(res),color='red',ls='--')
    ax.set(xlabel='DSM - LAS surface Z (m)',ylabel='Count',title=title)
plt.tight_layout()
qa_path=outdir/'09_Yoko_LAS_DSM_registration_QA.png'
fig.savefig(qa_path,dpi=180,bbox_inches='tight')
plt.show()
summary={'LAS':str(las_path),'DSM':str(dsm_path),'dx_m':dx,'dy_m':dy,'optional_dz_m':dz,
         'apply_z':APPLY_Z,'mad_before_m':score(0,0),'mad_after_m':score(dx,dy),
         'matched_before':len(before),'matched_after':len(after),'grid_resolution_m':resolution,
         'warning':'Canopy height discrepancy may cause biased registration.'}
json_path=outdir/'09_Yoko_LAS_DSM_registration.json'
json_path.write_text(json.dumps(summary,indent=2))
print('QA:',qa_path,'\nParameters:',json_path)

In [ ]:
corrected_path=outdir/'Yoko_aligned.las'
if EXPORT_CORRECTED_LAS:
    if corrected_path.exists():
        raise FileExistsError(f'Output already exists: {corrected_path}. Inspect/remove deliberately.')
    if not np.isfinite(dx+dy+dz):
        raise RuntimeError('Invalid translation')
    with laspy.open(las_path) as src:
        hdr=copy.deepcopy(src.header)
        with laspy.open(corrected_path,mode='w',header=hdr) as dst:
            for chunk in src.chunk_iterator(CHUNK_POINTS):
                chunk.x=np.asarray(chunk.x)+dx
                chunk.y=np.asarray(chunk.y)+dy
                if APPLY_Z:
                    chunk.z=np.asarray(chunk.z)+dz
                dst.write_points(chunk)
    with laspy.open(corrected_path) as check:
        assert check.header.point_count==npoints
        print('Corrected LAS:',corrected_path,'points',check.header.point_count)
else:
    print('Export disabled')
print('IMPORTANT: inspect QA before accepting this registration.')

## Interpretation
A distinct score minimum, clear reduction in MAD, and broad overlap support a plausible translation. A flat/multiple minimum, edge-hitting optimum, or little improvement indicate weak evidence; do not treat the output as validated. The method assumes horizontal translation only and does not model rotation, local warping, or canopy-growth differences. Vertical correction is OFF by default.